# PMM Dynamic — Оптимизация и Forward Test
## ETH/USDT | Bybit Spot | Google Colab

Этот ноутбук реализует полный pipeline для стратегии PMM Dynamic:

| Этап | Описание |
|------|----------|
| **Данные** | Загрузка исторических свечей ETH/USDT через CCXT (Bybit spot) |
| **EDA** | Анализ волатильности (NATR) и трендового сигнала (MACD) |
| **Оптимизация** | Подбор параметров через Optuna (in-sample, 70% данных) |
| **Forward Test** | Проверка лучшей конфигурации на OOS (30% данных) |
| **Сохранение** | Лучший конфиг сохраняется на Google Drive в формате YAML |

> **Контроллер:** [pmm_dynamic.py](https://github.com/hummingbot/hummingbot/blob/master/controllers/market_making/pmm_dynamic.py)  
> **Логика:** NATR масштабирует спреды, MACD смещает reference price


---
## ⚙️ Шаг 1 — Установка зависимостей
> **После выполнения этой ячейки перезапустите runtime** (`Runtime → Restart session`)

In [ ]:
import os, sys, subprocess

def _run(cmd, check=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0 and check:
        print(result.stderr[-2000:])
    return result.returncode == 0

print("[1/4] Installing core packages...")
_run("pip install -q hummingbot", check=True)
_run("pip install -q pandas-ta ccxt optuna optuna-dashboard pyarrow plotly nest_asyncio pyyaml python-dotenv statsmodels motor asyncpg", check=True)

# hummingbot.client.settings.AllConnectorSettings.create_connector_settings() scans ALL
# connector directories at import time. injective_v2_utils imports pyinjective which causes:
#   TypeError: duplicate file name gogoproto/gogo.proto (protobuf pool conflict with Colab)
# AllConnectorSettings only catches ModuleNotFoundError — not TypeError.
# Fix: remove the injective_v2 directory so the import raises ModuleNotFoundError → skipped.
print("[2/4] Removing injective_v2 connector (protobuf conflict fix)...")
_run("find /usr/local/lib/python3.12/dist-packages/hummingbot/connector/exchange "
     "-maxdepth 1 -name 'injective_v2' -type d -exec rm -rf {} +")
_run("pip uninstall -y pyinjective")
result = subprocess.run(
    "find /usr/local/lib/python3.12/dist-packages/hummingbot/connector/exchange "
    "-maxdepth 1 -name 'injective_v2' -type d",
    shell=True, capture_output=True, text=True,
)
if result.stdout.strip():
    print("  ⚠️  injective_v2 still present — restart will clear it")
else:
    print("  ✅ injective_v2 removed")

print("[3/4] Cloning quants-lab...")
if not os.path.exists('/content/quants-lab'):
    _run("git clone -q https://github.com/Shmel999/quants-lab.git /content/quants-lab", check=True)
    print("  quants-lab cloned")
else:
    print("  quants-lab already present")

sys.path.insert(0, '/content/quants-lab')

print("[4/4] Done.")
print("⚠️  Restart the runtime now: Runtime → Restart session")

---
## 💾 Шаг 2 — Подключение Google Drive

In [ ]:
import os, sys

# Add quants-lab to Python path (after runtime restart)
if '/content/quants-lab' not in sys.path:
    sys.path.insert(0, '/content/quants-lab')

from google.colab import drive
drive.mount('/content/drive')
print("\u2705 Google Drive mounted at /content/drive")

---
## ⚙️ Шаг 3 — Параметры (редактировать здесь)

In [ ]:
# ─────────────── Trading parameters ───────────────
CONNECTOR        = "bybit"       # Hummingbot connector name
TRADING_PAIR     = "ETH-USDT"   # Hummingbot pair format
CCXT_SYMBOL      = "ETH/USDT"   # CCXT symbol format
INTERVAL         = "15m"          # Candle interval (must match PMM Dynamic config)
TOTAL_AMOUNT_QUOTE = 1000        # Capital per strategy in USDT

# ─────────────── Data range ───────────────
# 180 дней даёт ~86 400 свечей по 3m — достаточно для надёжной оптимизации MACD/NATR.
# При 999 свечей/запрос это ~87 батчей (~10 секунд загрузки).
LOOKBACK_DAYS    = 180           # Total historical days to download
TRAIN_RATIO      = 0.70          # 70% in-sample (~126 дней), 30% forward test (~54 дня)

# ─────────────── Optimization ───────────────
N_TRIALS         = 100           # Optuna trials (increase for better results)
STUDY_NAME       = "pmm_dynamic_bybit_eth_usdt"
TRADE_COST       = 0.0006        # 0.06% round-trip (Bybit spot maker fee)
# BT_RESOLUTION must match INTERVAL: BacktestingEngine loads candles from the parquet cache
# (bybit|ETH-USDT|3m.parquet). Using a different resolution (e.g. "1m") would cause the
# engine to attempt a live Bybit API call and fail with IndexError on empty response.
BT_RESOLUTION    = INTERVAL      # Keep in sync with INTERVAL

# ─────────────── Google Drive paths ───────────────
DRIVE_BASE       = "/content/drive/MyDrive/quants-lab"
DRIVE_CANDLES    = f"{DRIVE_BASE}/candles"
DRIVE_STUDIES    = f"{DRIVE_BASE}/studies"
DRIVE_CONFIGS    = f"{DRIVE_BASE}/configs"

# ─────────────── Derived ───────────────
import time as _time
END_TS    = _time.time() - 3600          # 1h buffer from now
START_TS  = END_TS - LOOKBACK_DAYS * 86400
SPLIT_TS  = START_TS + (END_TS - START_TS) * TRAIN_RATIO

import pandas as _pd
_candles_per_day = 24 * 60 // int(INTERVAL.replace("m", ""))
_total_candles   = LOOKBACK_DAYS * _candles_per_day
_batches         = -(-_total_candles // 999)   # ceiling division

print("Configuration:")
print(f"  Pair      : {TRADING_PAIR} on {CONNECTOR} (spot)")
print(f"  Interval  : {INTERVAL}  ({_candles_per_day} свечей/день)")
print(f"  Capital   : ${TOTAL_AMOUNT_QUOTE:,} USDT")
print(f"  Lookback  : {LOOKBACK_DAYS} days  (~{_total_candles:,} candles, ~{_batches} batches)")
print(f"  IS period : {_pd.to_datetime(START_TS, unit='s').date()} → {_pd.to_datetime(SPLIT_TS, unit='s').date()} ({int(LOOKBACK_DAYS * TRAIN_RATIO)} days)")
print(f"  OOS period: {_pd.to_datetime(SPLIT_TS, unit='s').date()} → {_pd.to_datetime(END_TS, unit='s').date()} ({int(LOOKBACK_DAYS * (1 - TRAIN_RATIO))} days)")
print(f"  n_trials  : {N_TRIALS}")
print(f"  Trade cost: {TRADE_COST*100:.3f}%")
print(f"  BT resol. : {BT_RESOLUTION}")

---
## 📦 Шаг 4 — Импорты и настройка директорий

In [ ]:
import os
import sys
import uuid
import json
import time
import asyncio
import warnings
import datetime
from decimal import Decimal
from unittest.mock import MagicMock

import numpy as np
import pandas as pd
import pandas_ta as ta
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import nest_asyncio
import optuna
import yaml

nest_asyncio.apply()
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Plotly renderer for Google Colab
pio.renderers.default = "colab"

# ── Stub out avellaneda Cython extension ────────────────────────────────────
# Import chain that triggers the conflict:
#   pmm_dynamic → MarketMakingControllerBase
#     → hummingbot.strategy_v2.controllers.__init__
#     → directional_trading_controller_base
#     → hummingbot.client.ui.__init__
#     → conf_migration
#     → avellaneda_market_making/__init__.py
#     → avellaneda_market_making.pyx (Cython) → scipy → numpy ABI mismatch
# Stubbing the .pyx module in sys.modules before any hummingbot import
# makes __init__.py's "from .avellaneda_market_making import ..." use the mock
# instead of compiling/loading the Cython extension.
sys.modules['hummingbot.strategy.avellaneda_market_making.avellaneda_market_making'] = MagicMock()

# ── Create Drive directories ──
for _path in [DRIVE_CANDLES, DRIVE_STUDIES, DRIVE_CONFIGS]:
    os.makedirs(_path, exist_ok=True)

# ── Runtime candles directory (BacktestingEngine reads from here) ──
RUNTIME_CANDLES_DIR = "/content/quants-lab/app/data/cache/candles"
os.makedirs(RUNTIME_CANDLES_DIR, exist_ok=True)

# ── Optuna SQLite on Drive (persists between sessions) ──
OPTUNA_STORAGE = f"sqlite:///{DRIVE_STUDIES}/{STUDY_NAME}.db"

# ── Candle file paths ──
CANDLE_FILENAME      = f"{CONNECTOR}|{TRADING_PAIR}|{INTERVAL}.parquet"
DRIVE_CANDLE_PATH    = f"{DRIVE_CANDLES}/{CANDLE_FILENAME}"
RUNTIME_CANDLE_PATH  = f"{RUNTIME_CANDLES_DIR}/{CANDLE_FILENAME}"

print("✅ Imports loaded")
print(f"  numpy  : {np.__version__}")
print(f"  Drive candles : {DRIVE_CANDLE_PATH}")
print(f"  Drive studies : {OPTUNA_STORAGE}")
print(f"  Drive configs : {DRIVE_CONFIGS}")
print(f"  Runtime cache : {RUNTIME_CANDLE_PATH}")

---
## 📥 Шаг 5 — Загрузка данных через CCXT (Bybit Spot)

In [ ]:
import ccxt
import math
import random

BATCH_SIZE    = 500    # conservative (Bybit max 1000, Binance max 1000)
MAX_RETRIES   = 2      # per-batch attempts PER exchange — fail fast, switch sooner
MAX_WAIT_S    = 30     # max wait per retry (seconds); avoids the 300s hang on geo-blocks
RATE_LIMIT_MS = 500    # ms between requests

# Ordered list of exchanges to try.
# Colab IPs are often geo-blocked by Bybit/Binance (403 → RateLimitExceeded).
# Kraken returns only recent ~500 candles regardless of `since` (no deep history via free API).
# Gate.io and KuCoin work from Colab and provide full historical depth.
FALLBACK_EXCHANGES = ["bybit", "binance", "gateio", "kucoin", "okx", "kraken"]


def _interval_to_ms(interval: str) -> int:
    units = {"m": 60, "h": 3600, "d": 86400}
    return int(interval[:-1]) * units[interval[-1]] * 1000


def _make_exchange(exchange_id: str):
    cls = getattr(ccxt, exchange_id, None)
    if cls is None:
        raise ValueError(f"{exchange_id} not found in ccxt")
    return cls({"enableRateLimit": True, "rateLimit": RATE_LIMIT_MS})


def _download_from_exchange(exchange_id, symbol, timeframe, since_ms, end_ms,
                             batch_starts, total_expected):
    """Download all batches from ONE exchange. Returns list of OHLCV rows or raises RuntimeError.

    Key safeguard: detects exchanges that ignore `since` and return the same recent candles
    for every batch (e.g. Kraken free API). If `last_ts_ms` doesn't advance from one batch
    to the next, a RuntimeError is raised immediately to switch to the next exchange.
    """
    exchange = _make_exchange(exchange_id)
    interval_ms = _interval_to_ms(timeframe)
    total_batches = len(batch_starts)
    all_candles = []
    prev_last_ts_ms = None   # tracks progress — detects stuck exchanges

    for batch_num, batch_since in enumerate(batch_starts, start=1):
        batch_ok = False
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                kwargs = dict(symbol=symbol, timeframe=timeframe,
                              since=batch_since, limit=BATCH_SIZE)
                if exchange_id == "bybit":
                    kwargs["params"] = {"category": "spot"}
                batch_data = exchange.fetch_ohlcv(**kwargs)
                batch_ok = True
                break
            except (ccxt.ExchangeNotAvailable, ccxt.AuthenticationError, ccxt.BadRequest) as e:
                # Non-retriable: geo-block, maintenance, bad endpoint → switch exchange immediately
                raise RuntimeError(
                    f"[{exchange_id}] non-retriable on batch {batch_num}: "
                    f"{type(e).__name__}"
                ) from e
            except (ccxt.RateLimitExceeded, ccxt.NetworkError, ccxt.RequestTimeout,
                    ccxt.DDoSProtection) as e:
                wait = min(10 * (2 ** attempt) + random.uniform(0, 3), MAX_WAIT_S)
                print(f"\n  ⏳ [{exchange_id}] батч {batch_num}, попытка {attempt}/{MAX_RETRIES}"
                      f" — ждём {wait:.0f}s  [{type(e).__name__}]")
                time.sleep(wait)
            except Exception as exc:
                print(f"\n  ⚠️  [{exchange_id}] батч {batch_num}, попытка {attempt}: {exc}")
                if attempt == MAX_RETRIES:
                    break
                time.sleep(5)

        if not batch_ok:
            raise RuntimeError(
                f"[{exchange_id}] батч {batch_num} не загружен после {MAX_RETRIES} попыток."
            )
        if not batch_data:
            print(f"\n  Пустой ответ на батче {batch_num}/{total_batches}. Стоп.")
            break

        last_ts_ms = batch_data[-1][0]

        # ── Stuck-exchange detection ────────────────────────────────────────────────
        # If last_ts_ms didn't advance from the previous batch, the exchange is ignoring
        # the `since` parameter and returning the same recent candles every time.
        # (Observed on Kraken free API: always returns the latest ~500 candles.)
        if prev_last_ts_ms is not None and last_ts_ms <= prev_last_ts_ms + interval_ms // 2:
            raise RuntimeError(
                f"[{exchange_id}] данные не продвигаются после батча {batch_num} "
                f"(stuck at {pd.to_datetime(last_ts_ms, unit='ms').date()}) — "
                f"биржа не поддерживает запрошенную глубину истории через бесплатный API."
            )
        prev_last_ts_ms = last_ts_ms
        # ───────────────────────────────────────────────────────────────────────────

        all_candles.extend(c for c in batch_data if c[0] <= end_ms)

        pct = min(len(all_candles) / total_expected * 100, 100)
        print(
            f"  Батч {batch_num:>3}/{total_batches}  |  "
            f"{len(all_candles):>7,} свечей  |  "
            f"{pct:5.1f}%  |  "
            f"до {pd.to_datetime(last_ts_ms, unit='ms').strftime('%Y-%m-%d')}",
            end="\r",
        )

        if last_ts_ms >= end_ms:
            break

        time.sleep(exchange.rateLimit / 1000)

    return all_candles


def download_candles_ccxt(
    exchange_id: str,
    symbol: str,
    timeframe: str,
    start_ts: float,
    end_ts: float,
    fallback_exchanges: list = None,
) -> "pd.DataFrame":
    """
    Загружает OHLCV батчами. Автоматически перебирает все биржи из списка.

    Если первичная биржа гео-заблокирована (Colab IP) или не поддерживает
    глубокую историю, немедленно переключается на следующую.
    """
    if fallback_exchanges is None:
        fallback_exchanges = FALLBACK_EXCHANGES

    exchanges_order = [exchange_id] + [e for e in fallback_exchanges if e != exchange_id]

    since_ms    = int(start_ts * 1000)
    end_ms      = int(end_ts   * 1000)
    interval_ms = _interval_to_ms(timeframe)

    batch_starts = []
    t = since_ms
    while t < end_ms:
        batch_starts.append(t)
        t += BATCH_SIZE * interval_ms

    total_batches  = len(batch_starts)
    total_expected = math.ceil((end_ms - since_ms) / interval_ms)

    print(f"Downloading {symbol} [{timeframe}]")
    print(f"  Period  : {pd.to_datetime(start_ts, unit='s').date()} → "
          f"{pd.to_datetime(end_ts, unit='s').date()}")
    print(f"  Expected: ~{total_expected:,} candles | {total_batches} batches × {BATCH_SIZE}")
    print(f"  Trying  : {' → '.join(exchanges_order)}")

    last_error = None
    for exch_id in exchanges_order:
        print(f"\n📡 [{exch_id}] connecting...", end=" ", flush=True)
        try:
            all_candles = _download_from_exchange(
                exch_id, symbol, timeframe, since_ms, end_ms, batch_starts, total_expected
            )
        except RuntimeError as e:
            print(f"failed ❌  ({e})")
            last_error = e
            continue
        except Exception as e:
            print(f"failed ❌  ({type(e).__name__}: {str(e)[:80]})")
            last_error = e
            continue

        if all_candles:
            print(f"\n✅ Загружено {len(all_candles):,} свечей с {exch_id}")
            df = pd.DataFrame(
                all_candles,
                columns=["_ts_ms", "open", "high", "low", "close", "volume"],
            )
            df["timestamp"]              = df["_ts_ms"] / 1000
            df["quote_asset_volume"]     = df["volume"] * df["close"]
            df["n_trades"]               = 0
            df["taker_buy_base_volume"]  = 0.0
            df["taker_buy_quote_volume"] = 0.0
            return (
                df.drop(columns=["_ts_ms"])
                  .drop_duplicates(subset=["timestamp"])
                  .sort_values("timestamp")
                  .reset_index(drop=True)
            )
        else:
            print(f"empty response ⚠️")
            last_error = RuntimeError(f"[{exch_id}] empty response")

    tried = ", ".join(exchanges_order)
    raise RuntimeError(
        f"Свечи не загружены ни с одного источника ({tried}).\n"
        f"  Последняя ошибка: {last_error}\n"
        f"  → Используйте ячейку 'Ручная загрузка' (Шаг 5b) и загрузите файл вручную."
    )


print("✅ CCXT downloader ready")
print(f"  batch={BATCH_SIZE}, retries={MAX_RETRIES}/exch, maxWait={MAX_WAIT_S}s")
print(f"  exchange order: {' → '.join(FALLBACK_EXCHANGES)}")
print(f"  stuck-data detection: ON (detects exchanges ignoring `since` param)")


In [ ]:
# ── Download or load from Drive cache ─────────────────────────────────────

need_download = True
_existing_df  = None   # сохраняем для фоллбэка при ошибке

if os.path.exists(DRIVE_CANDLE_PATH):
    print(f"📂 Cache found on Drive: {DRIVE_CANDLE_PATH}")
    _existing_df  = pd.read_parquet(DRIVE_CANDLE_PATH)
    cached_start  = _existing_df["timestamp"].min()
    cached_end    = _existing_df["timestamp"].max()

    covers_start = cached_start <= START_TS + 3600         # допуск 1h на начало
    covers_end   = cached_end   >= END_TS   - 86400        # допуск 24h на конец

    if covers_start and covers_end:
        print(f"✅ Cache is current ({len(_existing_df):,} candles). Skipping download.")
        print(f"   cached: {pd.to_datetime(cached_start, unit='s').date()} → "
              f"{pd.to_datetime(cached_end, unit='s').date()}")
        candles_df    = _existing_df
        need_download = False

    elif covers_start and not covers_end:
        # ── Incremental tail update: качаем только хвост ──────────────
        gap_h    = (END_TS - cached_end) / 3600
        tail_from = pd.to_datetime(cached_end, unit='s').strftime('%Y-%m-%d %H:%M')
        tail_to   = pd.to_datetime(END_TS,     unit='s').strftime('%Y-%m-%d')
        print(f"⚡ Incremental update — downloading tail only "
              f"({tail_from} → {tail_to}, gap {gap_h:.1f}h)")
        tail_df   = download_candles_ccxt(
            exchange_id=CONNECTOR,
            symbol=CCXT_SYMBOL,
            timeframe=INTERVAL,
            start_ts=cached_end,
            end_ts=END_TS,
        )
        candles_df = (
            pd.concat([_existing_df, tail_df])
              .drop_duplicates(subset=["timestamp"])
              .sort_values("timestamp")
              .reset_index(drop=True)
        )
        candles_df.to_parquet(DRIVE_CANDLE_PATH, index=False)
        print(f"✅ Cache updated on Drive: {DRIVE_CANDLE_PATH}")
        need_download = False

    else:
        # Нужна полная или частичная перезагрузка
        _missing = []
        if not covers_start:
            _missing.append(
                f"начало: нужно {pd.to_datetime(START_TS, unit='s').date()}, "
                f"есть {pd.to_datetime(cached_start, unit='s').date()}"
            )
        if not covers_end:
            _missing.append(
                f"конец: нужно {pd.to_datetime(END_TS, unit='s').strftime('%Y-%m-%d %H:%M')}, "
                f"есть {pd.to_datetime(cached_end, unit='s').strftime('%Y-%m-%d %H:%M')} "
                f"(разрыв {(END_TS - cached_end)/3600:.1f}h > 24h допуска)"
            )
        print(f"⚠️  Cache outdated ({'; '.join(_missing)}) — re-downloading...")

if need_download:
    try:
        candles_df = download_candles_ccxt(
            exchange_id=CONNECTOR,
            symbol=CCXT_SYMBOL,
            timeframe=INTERVAL,
            start_ts=START_TS,
            end_ts=END_TS,
        )
        candles_df.to_parquet(DRIVE_CANDLE_PATH, index=False)
        print(f"✅ Saved to Drive: {DRIVE_CANDLE_PATH}")

    except RuntimeError as _download_err:
        # ── Graceful fallback: используем кэш даже если он устарел ───
        if _existing_df is not None:
            print(f"\n⚠️  Download failed — falling back to cached data.")
            print(f"   Error: {_download_err}")
            candles_df   = _existing_df
            actual_start = candles_df["timestamp"].min()
            actual_end   = candles_df["timestamp"].max()
            _days_avail  = (actual_end - actual_start) / 86400
            print(f"   Cached range  : {pd.to_datetime(actual_start, unit='s').date()} → "
                  f"{pd.to_datetime(actual_end, unit='s').date()} ({_days_avail:.0f} days)")
            if _days_avail < 30:
                print("   ⚠️  Кэш слишком короткий (<30 дней) — результаты оптимизации будут ненадёжными!")
            # Корректируем глобальные переменные под доступный диапазон
            START_TS = actual_start
            END_TS   = actual_end
            SPLIT_TS = START_TS + (END_TS - START_TS) * TRAIN_RATIO
            print(f"   Analysis adjusted → IS: {pd.to_datetime(START_TS, unit='s').date()} → "
                  f"{pd.to_datetime(SPLIT_TS, unit='s').date()}, "
                  f"OOS: {pd.to_datetime(SPLIT_TS, unit='s').date()} → "
                  f"{pd.to_datetime(END_TS, unit='s').date()}")
        else:
            print("\n❌ Нет ни кэша, ни успешной загрузки. Возможные причины:")
            print("   • Bybit geo-блокирует Colab-IP (попробуйте позже или переключитесь на другой runtime)")
            print("   • Нет соединения с интернетом")
            raise

# ── Copy to runtime dir so BacktestingEngine can find it ──────────
candles_df.to_parquet(RUNTIME_CANDLE_PATH, index=False)
print(f"✅ Copied to runtime: {RUNTIME_CANDLE_PATH}")

# ── Summary ──
_dur_days = (candles_df["timestamp"].max() - candles_df["timestamp"].min()) / 86400
print(f"\nData summary:")
print(f"  Candles  : {len(candles_df):,}")
print(f"  Duration : {_dur_days:.1f} days")
print(f"  From     : {pd.to_datetime(candles_df['timestamp'].min(), unit='s')}")
print(f"  To       : {pd.to_datetime(candles_df['timestamp'].max(), unit='s')}")
print(f"  Close    : ${candles_df['close'].min():.2f} — ${candles_df['close'].max():.2f}")
print(f"  Avg vol  : {candles_df['volume'].mean():.2f} ETH / candle")
print(f"  File size: {os.path.getsize(DRIVE_CANDLE_PATH) / 1024:.0f} KB")


---
## 📤 Шаг 5b — Ручная загрузка (если все биржи заблокировали Colab IP)

> **Запускайте эту ячейку только если Шаг 5 завершился ошибкой** (все биржи недоступны)

**Как получить файл с данными:**

Файл `bybit|ETH-USDT|15m.parquet` уже скачан на сервере quants-lab.  
Скачайте его по одному из способов:

| Способ | Команда / действие |
|--------|-------------------|
| **VS Code / quants-lab IDE** | Открыть `research_notebooks/eda_strategies/pmm_dynamic/data/bybit\|ETH-USDT\|15m.parquet` → ПКМ → Download |
| **scp / rsync** | `scp user@server:/home/coder/quants-lab/research_notebooks/eda_strategies/pmm_dynamic/data/'bybit\|ETH-USDT\|15m.parquet' .` |
| **Colab upload** | Запустите ячейку ниже и загрузите файл через диалог |

После загрузки файл автоматически копируется на Drive и в runtime-кэш.

In [ ]:
# ── Ручная загрузка файла (запускать только если Шаг 5 завершился ошибкой) ──
#
# 1. Скачайте файл с сервера quants-lab:
#      research_notebooks/eda_strategies/pmm_dynamic/data/bybit|ETH-USDT|15m.parquet
# 2. Запустите эту ячейку — появится диалог выбора файла
# 3. Выберите скачанный .parquet файл
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import files
import shutil

print("📂 Выберите файл bybit|ETH-USDT|15m.parquet ...")
uploaded = files.upload()

if not uploaded:
    print("⚠️  Файл не выбран. Пропускаем.")
else:
    uploaded_name = list(uploaded.keys())[0]
    if not uploaded_name.endswith(".parquet"):
        print(f"⚠️  Ожидался .parquet файл, получен: {uploaded_name}")
    else:
        # Save uploaded bytes to expected filename
        target_name = CANDLE_FILENAME  # e.g. bybit|ETH-USDT|15m.parquet
        with open(target_name, "wb") as f:
            f.write(uploaded[uploaded_name])

        # Validate
        _df_check = pd.read_parquet(target_name)
        _days = (_df_check["timestamp"].max() - _df_check["timestamp"].min()) / 86400
        print(f"✅ Загружено: {len(_df_check):,} свечей, "
              f"{pd.to_datetime(_df_check['timestamp'].min(), unit='s').date()} → "
              f"{pd.to_datetime(_df_check['timestamp'].max(), unit='s').date()} "
              f"({_days:.0f} дней)")

        if _days < 30:
            print("⚠️  Файл слишком короткий (<30 дней) — оптимизация будет ненадёжной.")

        # Copy to Drive cache (persistent)
        os.makedirs(DRIVE_CANDLES, exist_ok=True)
        shutil.copy(target_name, DRIVE_CANDLE_PATH)
        print(f"✅ Сохранено на Drive: {DRIVE_CANDLE_PATH}")

        # Copy to runtime cache (BacktestingEngine reads from here)
        shutil.copy(target_name, RUNTIME_CANDLE_PATH)
        print(f"✅ Скопировано в runtime: {RUNTIME_CANDLE_PATH}")

        # Set candles_df so subsequent cells work normally
        candles_df = _df_check
        START_TS = candles_df["timestamp"].min()
        END_TS   = candles_df["timestamp"].max()
        SPLIT_TS = START_TS + (END_TS - START_TS) * TRAIN_RATIO

        print(f"\nData summary:")
        print(f"  Candles  : {len(candles_df):,}")
        print(f"  Duration : {_days:.1f} days")
        print(f"  IS period: {pd.to_datetime(START_TS, unit='s').date()} → "
              f"{pd.to_datetime(SPLIT_TS, unit='s').date()}")
        print(f"  OOS period:{pd.to_datetime(SPLIT_TS, unit='s').date()} → "
              f"{pd.to_datetime(END_TS, unit='s').date()}")
        print("\n✅ Готово — продолжайте с Шага 6 (EDA)")

KeyboardInterrupt: 

---
## 📊 Шаг 6 — EDA: Цена и Объём

In [ ]:
df = candles_df.copy()
df.index = pd.to_datetime(df["timestamp"], unit="s")

# Resample to 1h for cleaner chart
df_1h = df.resample("1h").agg({
    "open":   "first",
    "high":   "max",
    "low":    "min",
    "close":  "last",
    "volume": "sum",
}).dropna()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(f"{TRADING_PAIR} Price (1h resampled)", "Volume (1h)"),
    row_heights=[0.70, 0.30], vertical_spacing=0.04,
)

# Candlestick
fig.add_trace(go.Candlestick(
    x=df_1h.index,
    open=df_1h["open"], high=df_1h["high"],
    low=df_1h["low"],   close=df_1h["close"],
    name="OHLC",
    increasing_line_color="#26a69a",
    decreasing_line_color="#ef5350",
), row=1, col=1)

# IS / OOS separator
split_str = pd.to_datetime(SPLIT_TS, unit="s").strftime("%Y-%m-%d %H:%M:%S")
fig.add_vline(x=split_str, line_width=2, line_dash="dash", line_color="yellow")
fig.add_annotation(
    x=split_str, y=1.02, xref="x", yref="paper",
    text="IS | OOS", showarrow=False,
    font=dict(color="yellow", size=11), xanchor="center",
)

# Volume bars
bar_colors = ["#26a69a" if c >= o else "#ef5350"
              for c, o in zip(df_1h["close"], df_1h["open"])]
fig.add_trace(go.Bar(
    x=df_1h.index, y=df_1h["volume"],
    marker_color=bar_colors, name="Volume",
), row=2, col=1)

fig.update_layout(
    template="plotly_dark", height=650,
    title=f"{TRADING_PAIR} | {CONNECTOR} Spot | Last {LOOKBACK_DAYS} days",
    xaxis_rangeslider_visible=False,
    showlegend=False,
)
fig.show()

# Basic stats
returns = df["close"].pct_change().dropna()
ann_vol = returns.std() * (525600 / 3) ** 0.5  # 3m candles → annual
print(f"ETH/USDT stats over last {LOOKBACK_DAYS} days:")
print(f"  Price range   : ${df['close'].min():.2f} — ${df['close'].max():.2f}")
print(f"  Annualised vol: {ann_vol*100:.1f}%")
print(f"  Avg 3m return : {returns.mean()*100:.4f}%")
print(f"  3m vol (mean) : {returns.std()*100:.4f}%")

---
## 📊 Шаг 7 — EDA: NATR и MACD (индикаторы PMM Dynamic)

In [ ]:
# Default PMM Dynamic parameter values (for EDA visualisation)
_MACD_FAST   = 21
_MACD_SLOW   = 42
_MACD_SIGNAL = 9
_NATR_LEN    = 14

df = candles_df.copy()
df.index = pd.to_datetime(df["timestamp"], unit="s")

# ── Compute indicators (mirrors PMMDynamicController.update_processed_data) ──
natr = ta.natr(df["high"], df["low"], df["close"], length=_NATR_LEN) / 100

macd_out  = ta.macd(df["close"], fast=_MACD_FAST, slow=_MACD_SLOW, signal=_MACD_SIGNAL)
macd_col  = f"MACD_{_MACD_FAST}_{_MACD_SLOW}_{_MACD_SIGNAL}"
macdh_col = f"MACDh_{_MACD_FAST}_{_MACD_SLOW}_{_MACD_SIGNAL}"

macd_line    = macd_out[macd_col]
macdh        = macd_out[macdh_col]
macd_signal  = -(macd_line - macd_line.mean()) / macd_line.std()   # z-score, inverted
macdh_signal = macdh.apply(lambda x: 1.0 if x > 0 else -1.0)

max_shift    = natr / 2
price_mult   = 0.5 * macd_signal + 0.5 * macdh_signal
ref_price    = df["close"] * (1 + price_mult * max_shift)
spread_upper = ref_price * (1 + natr * 1)   # 1× NATR spread example
spread_lower = ref_price * (1 - natr * 1)

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=(
        "Price, Reference Price & Example Spread Band",
        "NATR — Normalized ATR (волатильность, %)",
        "MACD Histogram",
    ),
    row_heights=[0.50, 0.25, 0.25], vertical_spacing=0.05,
)

# ── Row 1: price + reference price + spread band ──
fig.add_trace(go.Scatter(
    x=df.index, y=df["close"], name="Close",
    line=dict(color="#00b4d8", width=1)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df.index, y=ref_price, name="Reference Price",
    line=dict(color="#ff9f1c", width=1.5, dash="dot")
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df.index, y=spread_upper, name="Spread upper (1×NATR)",
    line=dict(color="rgba(50,200,50,0.4)", width=1)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df.index, y=spread_lower, name="Spread lower (1×NATR)",
    fill="tonexty", fillcolor="rgba(50,200,50,0.05)",
    line=dict(color="rgba(50,200,50,0.4)", width=1)
), row=1, col=1)
split_str = pd.to_datetime(SPLIT_TS, unit="s").strftime("%Y-%m-%d %H:%M:%S")
fig.add_vline(x=split_str, line_dash="dash", line_color="yellow", line_width=1.5)
fig.add_annotation(
    x=split_str, y=1.02, xref="x", yref="paper",
    text="IS | OOS", showarrow=False,
    font=dict(color="yellow", size=11), xanchor="center",
)

# ── Row 2: NATR ──
fig.add_trace(go.Scatter(
    x=df.index, y=natr * 100, name="NATR %",
    line=dict(color="#8338ec", width=1.5),
    fill="tozeroy", fillcolor="rgba(131,56,236,0.15)"
), row=2, col=1)

# ── Row 3: MACD histogram ──
bar_colors = ["#26a69a" if v >= 0 else "#ef5350" for v in macdh.fillna(0)]
fig.add_trace(go.Bar(
    x=df.index, y=macdh, name="MACD Hist",
    marker_color=bar_colors
), row=3, col=1)

fig.update_layout(
    template="plotly_dark", height=850, showlegend=True,
    title=f"PMM Dynamic Indicators — {TRADING_PAIR} {INTERVAL} | MACD({_MACD_FAST},{_MACD_SLOW},{_MACD_SIGNAL}) NATR({_NATR_LEN})",
)
fig.show()

print("NATR stats:")
print(f"  mean : {natr.mean()*100:.3f}%")
print(f"  p50  : {natr.median()*100:.3f}%")
print(f"  p95  : {natr.quantile(0.95)*100:.3f}%")
print(f"  max  : {natr.max()*100:.3f}%")
print(f"\nAverage |price shift| from MACD: {abs(ref_price / df['close'] - 1).mean()*100:.4f}%")

---
## 🤖 Шаг 8 — Config Generator для Optuna

Пространство поиска:

| Параметр | Диапазон | Описание |
|----------|----------|----------|
| `macd_fast` | 8–21 | Быстрое окно MACD |
| `macd_slow_offset` | 5–30 (шаг 5) | Разрыв slow–fast, slow = fast + offset |
| `macd_signal` | 7–14 | Сигнальная линия MACD |
| `natr_length` | {7, 10, 14, 21} | Период NATR |
| `n_levels` | 2–4 | Количество уровней ордеров |
| `spread_start` | 0.5–2.5× NATR | Первый уровень спреда |
| `spread_step` | 0.5–2.0× NATR | Шаг между уровнями |
| `take_profit` | 0.5%–5% | Take profit |
| `stop_loss` | 0.5%–3% | Stop loss |
| `time_limit` | 4h–48h | Максимальное время жизни позиции |
| `executor_refresh_time` | 1–10 мин | Частота обновления ордеров |
| `cooldown_time` | 1–10 мин | Пауза после закрытия позиции |

In [ ]:
from app.controllers.market_making.pmm_dynamic import PMMDynamicControllerConfig
from core.backtesting.optimizer import BacktestingConfig, BaseStrategyConfigGenerator

try:
    from hummingbot.strategy_v2.utils.distributions import Distributions
except ImportError:
    # Fallback if distributions module unavailable
    class Distributions:
        @staticmethod
        def arithmetic(n, start, step):
            return [round(start + i * step, 4) for i in range(n)]


class PMMDynamicConfigGenerator(BaseStrategyConfigGenerator):
    """
    Optuna config generator for PMM Dynamic (market making).

    PMM Dynamic logic:
      - reference_price = close * (1 + macd_shift * natr/2)
      - spread = spread_multiplier * natr   (spread_multiplier = buy/sell_spreads[level])
    """

    async def generate_config(self, trial) -> BacktestingConfig:
        # ── MACD ──────────────────────────────────────────────────────
        macd_fast        = trial.suggest_int("macd_fast", 8, 21, step=1)
        macd_slow_offset = trial.suggest_int("macd_slow_offset", 5, 30, step=5)
        macd_slow        = macd_fast + macd_slow_offset   # always > fast
        macd_signal      = trial.suggest_int("macd_signal", 7, 14, step=1)

        # ── NATR ──────────────────────────────────────────────────────
        natr_length = trial.suggest_categorical("natr_length", [7, 10, 14, 21])

        # ── Spread levels (in units of NATR) ──────────────────────────
        n_levels     = trial.suggest_int("n_levels", 2, 4)
        spread_start = trial.suggest_float("spread_start", 0.5, 2.5, step=0.5)
        spread_step  = trial.suggest_float("spread_step",  0.5, 2.0, step=0.5)
        spreads      = [round(spread_start + i * spread_step, 2) for i in range(n_levels)]
        amounts      = Distributions.arithmetic(n_levels, 0.01, 0.01)

        # ── Risk management ───────────────────────────────────────────
        take_profit = trial.suggest_float("take_profit", 0.005, 0.05,  step=0.005)
        stop_loss   = trial.suggest_float("stop_loss",   0.005, 0.03,  step=0.005)
        time_limit  = trial.suggest_int(  "time_limit",  3600 * 4, 3600 * 48, step=3600 * 4)

        # ── Timing ────────────────────────────────────────────────────
        executor_refresh_time = trial.suggest_int("executor_refresh_time", 60, 600, step=60)
        cooldown_time         = trial.suggest_int("cooldown_time",         60, 600, step=60)

        config = PMMDynamicControllerConfig(
            id=str(uuid.uuid4()),
            connector_name=self.config["connector_name"],
            trading_pair=self.config["trading_pair"],
            candles_connector=self.config["connector_name"],
            candles_trading_pair=self.config["trading_pair"],
            interval=self.config.get("interval", "3m"),
            total_amount_quote=Decimal(str(self.config.get("total_amount_quote", TOTAL_AMOUNT_QUOTE))),
            buy_spreads=spreads,
            sell_spreads=spreads,
            buy_amounts_pct=amounts,
            sell_amounts_pct=amounts,
            macd_fast=macd_fast,
            macd_slow=macd_slow,
            macd_signal=macd_signal,
            natr_length=natr_length,
            take_profit=Decimal(str(take_profit)),
            stop_loss=Decimal(str(stop_loss)),
            time_limit=time_limit,
            executor_refresh_time=executor_refresh_time,
            cooldown_time=cooldown_time,
            leverage=1,   # Spot trading
        )

        return BacktestingConfig(config=config, start=self.start, end=self.end)


print("\u2705 PMMDynamicConfigGenerator defined")
print(f"  Search space: {11} parameters")

---
## 🔧 Шаг 9 — Инициализация BacktestingEngine

In [ ]:
import types
import logging

from core.backtesting.engine import BacktestingEngine
from core.backtesting.optimizer import StrategyOptimizer

engine = BacktestingEngine(load_cached_data=True)

# ── Patch 1: skip _update_trading_rules (calls live GET /instruments-info) ──
# This is the method that makes the 403 HTTP call when Colab IP is geo-blocked.
async def _noop_trading_rules(connector_name: str) -> None:
    pass

engine._bt_engine.backtesting_data_provider.initialize_trading_rules = _noop_trading_rules

# ── Patch 2: pre-populate trading pair symbol map ────────────────────────────
# BacktestingEngine lazily creates a BybitExchange connector to resolve trading
# pair symbols (ETH-USDT → ETHUSDT). Without a map the connector calls the same
# blocked Bybit API, which raises KeyError: 'ETH-USDT' in get_last_traded_price().
# Pre-populating the bidict avoids the API call entirely.
try:
    from bidict import bidict as _bidict
    _bybit_conn = engine._bt_engine.backtesting_data_provider.connectors.get(CONNECTOR)
    if _bybit_conn is not None:
        # Bybit spot canonical symbol: ETH-USDT → ETHUSDT
        _bybit_conn._trading_pair_symbol_map = _bidict({
            TRADING_PAIR: TRADING_PAIR.replace("-", "")
        })
        # Also noop the initialiser so it doesn't overwrite our map later
        async def _noop_sym_map_init(self):
            pass
        _bybit_conn._initialize_trading_pair_symbol_map = types.MethodType(
            _noop_sym_map_init, _bybit_conn
        )
        print("  ✅ Trading pair symbol map pre-populated "
              f"({TRADING_PAIR} → {TRADING_PAIR.replace('-', '')})")
    else:
        print("  ⚠️  Could not access bybit connector for symbol map pre-population")
except Exception as _sym_err:
    print(f"  ⚠️  Symbol map patch skipped: {_sym_err}")

# ── Patch 3: silence the geo-block error logs (cosmetic) ─────────────────────
# These loggers produce ERROR-level noise when Colab is geo-blocked by CloudFront.
# The errors are caught internally and don't stop the backtesting.
for _noisy_logger in [
    "hummingbot.connector.time_synchronizer",
    "hummingbot.connector.exchange.bybit.bybit_exchange.BybitExchange",
    "hummingbot.data_feed.market_data_provider",
]:
    logging.getLogger(_noisy_logger).setLevel(logging.CRITICAL)

# ── Verify loaded feeds ───────────────────────────────────────────────────────
feeds = engine._bt_engine.backtesting_data_provider.candles_feeds
print("\n✅ BacktestingEngine ready")
print("Loaded feeds:")
for feed_key in feeds.keys():
    df_feed = feeds[feed_key]
    print(f"  [{feed_key}]: {len(df_feed):,} candles, "
          f"{pd.to_datetime(df_feed['timestamp'].min(), unit='s').date()} → "
          f"{pd.to_datetime(df_feed['timestamp'].max(), unit='s').date()}")


---
## 🎯 Шаг 10 — Оптимизация (In-Sample)

In [ ]:
# IS period boundaries
is_start_dt = datetime.datetime.fromtimestamp(START_TS, tz=datetime.timezone.utc)
is_end_dt   = datetime.datetime.fromtimestamp(SPLIT_TS, tz=datetime.timezone.utc)

print(f"In-sample : {is_start_dt.date()} → {is_end_dt.date()}")
print(f"Study     : {STUDY_NAME}")
print(f"Storage   : {OPTUNA_STORAGE}")
print(f"Trials    : {N_TRIALS}")

config_generator = PMMDynamicConfigGenerator(
    start_date=is_start_dt,
    end_date=is_end_dt,
    config={
        "connector_name"    : CONNECTOR,
        "trading_pair"      : TRADING_PAIR,
        "interval"          : INTERVAL,
        "total_amount_quote": TOTAL_AMOUNT_QUOTE,
    },
)

optimizer = StrategyOptimizer(
    storage_name=OPTUNA_STORAGE,
    load_cached_data=False,   # engine already loaded
    resolution=BT_RESOLUTION,
)
# Inject already-loaded engine to avoid re-loading candles
optimizer._backtesting_engine = engine

print("\nStarting optimization...")

async def _optimize():
    await optimizer.optimize(
        study_name=STUDY_NAME,
        config_generator=config_generator,
        n_trials=N_TRIALS,
        load_if_exists=True,
    )

asyncio.get_event_loop().run_until_complete(_optimize())
print("\n\u2705 Optimization complete!")

---
## 📈 Шаг 11 — Анализ результатов оптимизации

In [ ]:
study = optimizer.get_study(STUDY_NAME)
trials_df = study.trials_dataframe()
trials_df = trials_df.dropna(subset=["value"])

# Clean column names
trials_df.columns = [
    c.replace("user_attrs_", "").replace("params_", "") for c in trials_df.columns
]

completed = trials_df[trials_df["state"] == "COMPLETE"] if "state" in trials_df.columns else trials_df

print(f"Trials completed : {len(completed)}")
print(f"Best Sharpe      : {study.best_trial.value:.4f}")
print(f"\nBest parameters:")
best_params = study.best_params
for k, v in best_params.items():
    print(f"  {k:<25}: {v}")

# Derived slow value
best_macd_slow = best_params["macd_fast"] + best_params["macd_slow_offset"]
print(f"  {'macd_slow (derived)':<25}: {best_macd_slow}")

# ── Optimization history chart ──
fig = go.Figure()

# All trials
fig.add_trace(go.Scatter(
    x=completed["number"], y=completed["value"],
    mode="markers",
    marker=dict(
        color=completed["value"], colorscale="Viridis",
        size=6, showscale=True,
        colorbar=dict(title="Sharpe"),
    ),
    name="Trials",
))

# Best-so-far curve
fig.add_trace(go.Scatter(
    x=completed["number"],
    y=completed["value"].cummax(),
    mode="lines",
    line=dict(color="gold", width=2),
    name="Best so far",
))

fig.update_layout(
    template="plotly_dark", height=420,
    title=f"Optimization History — {STUDY_NAME}",
    xaxis_title="Trial #", yaxis_title="Sharpe Ratio (IS)",
)
fig.show()

# ── Top-10 by Sharpe ──
metric_cols = [c for c in [
    "value", "net_pnl", "net_pnl_quote", "sharpe_ratio", "profit_factor",
    "max_drawdown_pct", "total_executors",
    "macd_fast", "macd_slow_offset", "macd_signal", "natr_length",
    "n_levels", "spread_start", "spread_step", "take_profit", "stop_loss",
] if c in completed.columns]

print("\nTop-10 trials by Sharpe Ratio:")
print(completed.nlargest(10, "value")[metric_cols].to_string(index=False))

In [ ]:
# ── Parameter importance ──
try:
    importance = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame(list(importance.items()), columns=["parameter", "importance"])
    imp_df = imp_df.sort_values("importance", ascending=True)

    fig = go.Figure(go.Bar(
        x=imp_df["importance"], y=imp_df["parameter"],
        orientation="h",
        marker_color="#00b4d8",
    ))
    fig.update_layout(
        template="plotly_dark", height=400,
        title="Parameter Importance (Sharpe Ratio)",
        xaxis_title="Importance",
    )
    fig.show()
except Exception as e:
    print(f"Parameter importance not available: {e}")

---
## 🏆 Шаг 12 — Бэктест лучшего конфига (In-Sample)

In [ ]:
# Reconstruct best config from stored JSON
best_trial = study.best_trial
best_config_json = best_trial.user_attrs.get("config", "{}")
best_config_dict = json.loads(best_config_json)

try:
    best_config = PMMDynamicControllerConfig.model_validate(best_config_dict)
except Exception:
    # Fallback for older Pydantic versions
    best_config = PMMDynamicControllerConfig(**best_config_dict)

print("Best config (IS):")
print(f"  MACD       : ({best_config.macd_fast}, {best_config.macd_slow}, {best_config.macd_signal})")
print(f"  NATR len   : {best_config.natr_length}")
print(f"  Buy spreads: {best_config.buy_spreads}")
print(f"  Sell spreads:{best_config.sell_spreads}")
print(f"  TP / SL    : {float(best_config.take_profit)*100:.2f}% / {float(best_config.stop_loss)*100:.2f}%")
print(f"  Time limit : {best_config.time_limit / 3600:.1f}h")

async def _run_backtest(start_ts, end_ts):
    return await engine.run_backtesting(
        config=best_config,
        start=int(start_ts),
        end=int(end_ts),
        backtesting_resolution=BT_RESOLUTION,
        trade_cost=TRADE_COST,
    )

print("\nRunning IS backtest...")
is_result = asyncio.get_event_loop().run_until_complete(_run_backtest(START_TS, SPLIT_TS))

print("\n" + "═" * 60)
print("IN-SAMPLE RESULTS (best config)")
print("═" * 60)
print(is_result.get_results_summary())

fig = is_result.get_backtesting_figure()
fig.update_layout(title=f"In-Sample Backtest — {TRADING_PAIR} | Best Config")
fig.show()

---
## 🔮 Шаг 13 — Forward Test (Out-of-Sample)

In [ ]:
oos_start_dt = datetime.datetime.fromtimestamp(SPLIT_TS, tz=datetime.timezone.utc)
oos_end_dt   = datetime.datetime.fromtimestamp(END_TS,   tz=datetime.timezone.utc)

print(f"Forward test: {oos_start_dt.date()} → {oos_end_dt.date()}")
print("Running OOS backtest with the IS-optimal config (no re-fitting)...")

oos_result = asyncio.get_event_loop().run_until_complete(_run_backtest(SPLIT_TS, END_TS))

print("\n" + "═" * 60)
print("FORWARD TEST RESULTS (out-of-sample)")
print("═" * 60)
print(oos_result.get_results_summary())

fig = oos_result.get_backtesting_figure()
fig.update_layout(title=f"Forward Test (OOS) — {TRADING_PAIR} | Best Config")
fig.show()

---
## 📊 Шаг 14 — Сравнение IS vs OOS

In [ ]:
IS  = is_result.results
OOS = oos_result.results

METRICS = [
    ("net_pnl",          "Net PnL %",       100, True),
    ("net_pnl_quote",    "Net PnL ($)",      1,   True),
    ("sharpe_ratio",     "Sharpe Ratio",     1,   True),
    ("profit_factor",    "Profit Factor",    1,   True),
    ("max_drawdown_pct", "Max Drawdown %",   100, False),
    ("total_executors",  "Total Executors",  1,   True),
    ("accuracy_long",    "Accuracy Long %",  100, True),
    ("accuracy_short",   "Accuracy Short %", 100, True),
]

print(f"{'Metric':<22} {'In-Sample':>12} {'OOS/Forward':>13} {'Ratio OOS/IS':>14}")
print("-" * 65)
for key, label, mul, higher_is_better in METRICS:
    is_v  = IS.get(key, 0)  * mul
    oos_v = OOS.get(key, 0) * mul
    ratio = oos_v / is_v if is_v != 0 else float("nan")
    flag  = "✅" if (ratio >= 0.5 and higher_is_better) or (ratio <= 1.5 and not higher_is_better) else "⚠️"
    print(f"{label:<22} {is_v:>12.3f} {oos_v:>13.3f} {ratio:>13.2f}  {flag}")

# ── Bar chart ──
chart_metrics = ["Net PnL %", "Sharpe Ratio", "Profit Factor", "Max DD %"]
is_vals  = [IS.get("net_pnl",0)*100, IS.get("sharpe_ratio",0),
            IS.get("profit_factor",0), IS.get("max_drawdown_pct",0)*100]
oos_vals = [OOS.get("net_pnl",0)*100, OOS.get("sharpe_ratio",0),
            OOS.get("profit_factor",0), OOS.get("max_drawdown_pct",0)*100]

fig = go.Figure(data=[
    go.Bar(name="In-Sample",    x=chart_metrics, y=is_vals,  marker_color="#00b4d8"),
    go.Bar(name="Out-of-Sample",x=chart_metrics, y=oos_vals, marker_color="#ff9f1c"),
])
fig.update_layout(
    template="plotly_dark", barmode="group",
    title="IS vs OOS Performance Comparison",
    height=420, showlegend=True,
)
fig.show()

# ── Cumulative PnL overlay ──
is_executors  = is_result.executors
oos_executors = oos_result.executors

if is_executors and oos_executors:
    is_pnl  = pd.Series([float(e.net_pnl_quote) for e in is_executors],
                        index=pd.to_datetime([e.close_timestamp for e in is_executors], unit="s"))
    oos_pnl = pd.Series([float(e.net_pnl_quote) for e in oos_executors],
                        index=pd.to_datetime([e.close_timestamp for e in oos_executors], unit="s"))

    split_str2 = pd.to_datetime(SPLIT_TS, unit="s").strftime("%Y-%m-%d %H:%M:%S")

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=is_pnl.index, y=is_pnl.cumsum(),
        name="IS Cumulative PnL", line=dict(color="#00b4d8", width=2),
    ))
    fig2.add_trace(go.Scatter(
        x=oos_pnl.index, y=oos_pnl.cumsum(),
        name="OOS Cumulative PnL", line=dict(color="#ff9f1c", width=2),
    ))
    fig2.add_vline(x=split_str2, line_dash="dash", line_color="yellow", line_width=1.5)
    fig2.add_annotation(
        x=split_str2, y=1.02, xref="x", yref="paper",
        text="IS | OOS", showarrow=False,
        font=dict(color="yellow", size=11), xanchor="center",
    )
    fig2.update_layout(
        template="plotly_dark", height=380,
        title="Cumulative PnL (USDT)",
        yaxis_title="Cumulative PnL ($)",
    )
    fig2.show()

---
## 💾 Шаг 15 — Сохранение лучшего конфига на Google Drive

In [ ]:
# Build complete YAML config for Hummingbot deployment
macd_slow_derived = best_params["macd_fast"] + best_params["macd_slow_offset"]
n_levels_best     = best_params["n_levels"]
spread_start_best = best_params["spread_start"]
spread_step_best  = best_params["spread_step"]
spreads_best      = [round(spread_start_best + i * spread_step_best, 2) for i in range(n_levels_best)]

config_out = {
    # ── Controller identity ──
    "id"               : str(uuid.uuid4()),
    "controller_name"  : "pmm_dynamic",
    "controller_type"  : "market_making",

    # ── Exchange / pair ──
    "connector_name"       : CONNECTOR,
    "trading_pair"         : TRADING_PAIR,
    "candles_connector"    : CONNECTOR,
    "candles_trading_pair" : TRADING_PAIR,
    "interval"             : INTERVAL,
    "leverage"             : 1,

    # ── Capital ──
    "total_amount_quote": float(TOTAL_AMOUNT_QUOTE),

    # ── Strategy parameters ──
    "buy_spreads"  : spreads_best,
    "sell_spreads" : spreads_best,
    "macd_fast"    : best_params["macd_fast"],
    "macd_slow"    : macd_slow_derived,
    "macd_signal"  : best_params["macd_signal"],
    "natr_length"  : best_params["natr_length"],

    # ── Risk management ──
    "take_profit"          : best_params["take_profit"],
    "stop_loss"            : best_params["stop_loss"],
    "time_limit"           : best_params["time_limit"],
    "executor_refresh_time": best_params["executor_refresh_time"],
    "cooldown_time"        : best_params["cooldown_time"],

    # ── Optimization metadata ──
    "_optimization": {
        "study_name"         : STUDY_NAME,
        "n_trials"           : N_TRIALS,
        "trade_cost"         : TRADE_COST,
        "bt_resolution"      : BT_RESOLUTION,
        "is_period"          : f"{is_start_dt.date()} → {is_end_dt.date()}",
        "oos_period"         : f"{oos_start_dt.date()} → {oos_end_dt.date()}",
        "is_sharpe_ratio"    : round(float(IS.get("sharpe_ratio", 0)), 4),
        "oos_sharpe_ratio"   : round(float(OOS.get("sharpe_ratio", 0)), 4),
        "is_net_pnl_pct"     : round(float(IS.get("net_pnl", 0)) * 100, 3),
        "oos_net_pnl_pct"    : round(float(OOS.get("net_pnl", 0)) * 100, 3),
        "is_profit_factor"   : round(float(IS.get("profit_factor", 0)), 4),
        "oos_profit_factor"  : round(float(OOS.get("profit_factor", 0)), 4),
        "optimized_at"       : datetime.datetime.now(datetime.timezone.utc).isoformat(),
    },
}

# Save to Drive
timestamp_str   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
config_filename = f"{STUDY_NAME}_{timestamp_str}.yml"
config_path     = f"{DRIVE_CONFIGS}/{config_filename}"

with open(config_path, "w") as f:
    yaml.dump(config_out, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"\u2705 Config saved to Drive: {config_path}")
print("\nFile contents:")
print("-" * 60)
print(yaml.dump(config_out, default_flow_style=False, allow_unicode=True, sort_keys=False))

---
## ✅ Итоги

| Что сохранено | Путь |
|--------------|-------|
| Исторические свечи (Drive) | `MyDrive/quants-lab/candles/bybit\|ETH-USDT\|15m.parquet` |
| Исторические свечи (сервер) | `quants-lab/research_notebooks/eda_strategies/pmm_dynamic/data/bybit\|ETH-USDT\|15m.parquet` |
| Optuna study | `MyDrive/quants-lab/studies/pmm_dynamic_bybit_eth_usdt.db` |
| Лучший конфиг (YAML) | `MyDrive/quants-lab/configs/pmm_dynamic_bybit_eth_usdt_YYYYMMDD_HHMM.yml` |

### Если данные не загружаются автоматически
1. Скачайте готовый файл с сервера quants-lab:  
   `research_notebooks/eda_strategies/pmm_dynamic/data/bybit|ETH-USDT|15m.parquet`
2. Запустите **Шаг 5b** и загрузите файл через диалог

### Следующие шаги
1. **Деплой** — скопируйте YAML конфиг в `hummingbot/conf/controllers/` и запустите бота
2. **Повторная оптимизация** — увеличьте `N_TRIALS` и перезапустите ячейку оптимизации (study накапливается)
3. **Другие пары** — измените `TRADING_PAIR` и `CCXT_SYMBOL` в шаге 3
4. **Другой интервал** — попробуйте `INTERVAL = "1m"` или `"5m"` (нужно скачать новые свечи)

### Интерпретация результатов
- `Sharpe Ratio OOS / IS ≥ 0.5` — стратегия обобщается, не переобучена  
- `Profit Factor > 1.2` — стратегия зарабатывает в OOS периоде  
- `Max Drawdown OOS < 20%` — приемлемый риск